In [4]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix


def print_category_class_accuracy(
    y_true,
    y_pred,
    labels_csv,
    label_ids=None,
):
    """
    Prints per-class accuracy grouped by category.

    y_true:
        True encoded labels.

    y_pred:
        Predicted encoded labels.

    labels_csv:
        Path to labels.csv.

    label_ids:
        Optional list of original label IDs used by the model.
    """

    labels_df = pd.read_csv(labels_csv)

    # --------------------------------------------------------
    # Filter to labels used by this model
    # --------------------------------------------------------

    if label_ids is not None:
        labels_df = labels_df[
            labels_df["id"].astype(int).isin(label_ids)
        ].copy()

    labels_df = labels_df.sort_values("id")

    original_ids = (
        labels_df["id"]
        .astype(int)
        .tolist()
    )

    class_names = (
        labels_df["label"]
        .astype(str)
        .tolist()
    )

    categories = (
        labels_df["category"]
        .astype(str)
        .tolist()
    )

    # --------------------------------------------------------
    # Confusion matrix
    # --------------------------------------------------------

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=range(len(original_ids)),
    )

    # --------------------------------------------------------
    # Per-class accuracy
    # --------------------------------------------------------

    class_accuracy = np.divide(
        np.diag(cm),
        cm.sum(axis=1),
        out=np.zeros(len(class_names), dtype=float),
        where=cm.sum(axis=1) != 0,
    )

    # --------------------------------------------------------
    # Group by category
    # --------------------------------------------------------

    category_results = {}

    for category in dict.fromkeys(categories):

        category_indices = [
            i
            for i, c in enumerate(categories)
            if c == category
        ]

        category_results[category] = []

        for i in category_indices:

            correct = cm[i, i]
            total = cm[i].sum()

            accuracy = (
                correct / total
                if total > 0
                else 0.0
            )

            category_results[category].append({
                "id": original_ids[i],
                "label": class_names[i],
                "correct": correct,
                "total": total,
                "accuracy": accuracy,
            })

    # --------------------------------------------------------
    # Print results
    # --------------------------------------------------------

    for category, results in category_results.items():

        print("\n")
        print("=" * 60)
        print(f"{category}")
        print("=" * 60)

        total_correct = 0
        total_samples = 0

        for result in results:

            print(
                f"{result['label']:20s}: "
                f"{result['accuracy']:.2%} "
                f"({result['correct']}/{result['total']})"
            )

            total_correct += result["correct"]
            total_samples += result["total"]

        category_accuracy = (
            total_correct / total_samples
            if total_samples > 0
            else 0.0
        )

        print("-" * 60)

        print(
            f"Category accuracy: "
            f"{category_accuracy:.2%}"
        )

In [2]:
import sys
import torch
import pandas as pd

from importlib import reload
from pathlib import Path

# ============================================================
# PROJECT SETUP
# ============================================================

REPO_ROOT = Path(
    r"C:\Projects\signia-fsl-recognition"
).resolve()

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


# ============================================================
# LOAD TRAINING MODULE
# ============================================================

import src.training.trainer as trainer_module

trainer_module = reload(trainer_module)

TrainingConfig = trainer_module.TrainingConfig
train = trainer_module.train


# ============================================================
# PATHS
# ============================================================

LABELS_CSV = REPO_ROOT / "csv" / "labels.csv"

BASE_BUNDLE = (
    REPO_ROOT
    / "notebooks"
    / "02_dynamic"
    / "fsl_dataset_clean.pt"
)

MERGED_BUNDLE = (
    REPO_ROOT
    / "data"
    / "merged_dataset.pt"
)

MODEL_ROOT = (
    REPO_ROOT
    / "artifacts"
    / "models"
)


# ============================================================
# LOAD LABELS
# ============================================================

labels_df = pd.read_csv(LABELS_CSV)

print("Categories found:")

category_groups = (
    labels_df
    .groupby("category")
)

for category, group in category_groups:
    print(
        f"  {category:<20} "
        f"{len(group)} classes "
        f"IDs {group['id'].min()}-{group['id'].max()}"
    )

Categories found:
  CALENDAR             12 classes IDs 30-41
  COLOR                13 classes IDs 72-84
  DAYS                 10 classes IDs 42-51
  DRINK                10 classes IDs 95-104
  FAMILY               10 classes IDs 52-61
  FOOD                 10 classes IDs 85-94
  GREETING             10 classes IDs 0-9
  NUMBER               10 classes IDs 20-29
  RELATIONSHIPS        10 classes IDs 62-71
  SURVIVAL             10 classes IDs 10-19


In [3]:
from src.data_collection.dataset_builder import DatasetBuilder

builder = DatasetBuilder()

builder.print_summary()

X, y = builder.merge_with_bundle(BASE_BUNDLE)

torch.save(
    {
        "X": X,
        "y": y,
    },
    MERGED_BUNDLE,
)

print(f"\nSaved merged bundle → {MERGED_BUNDLE}")
print(f"Total samples: {len(X)}")


Label                          |   ID | Samples
--------------------------------------------------
GOOD MORNING                   |    0 |      39
GOOD AFTERNOON                 |    1 |      36
GOOD EVENING                   |    2 |      36
HELLO                          |    3 |      29
IM FINE                        |    5 |      25
--------------------------------------------------
TOTAL                          |      |     165

Loaded bundle: C:\Projects\signia-fsl-recognition\notebooks\02_dynamic\fsl_dataset_clean.pt
  Base dataset: 2129 samples, shape (2129, 30, 126)
Merged 165 collected samples into dataset. Total: 2294 samples.

Saved merged bundle → C:\Projects\signia-fsl-recognition\data\merged_dataset.pt
Total samples: 2294


In [5]:
# ============================================================
# TRAIN ALL CATEGORIES
# ============================================================

results = {}

for category, group in category_groups:

    category_upper = str(category).upper()
    category_lower = category_upper.lower()

    label_ids = (
        group["id"]
        .astype(int)
        .tolist()
    )

    print("\n")
    print("=" * 70)
    print(f"TRAINING CATEGORY: {category_upper}")
    print("=" * 70)

    print(
        f"Labels: {label_ids}"
    )

    print(
        "\nClasses:"
    )

    print(
        group[
            ["id", "label"]
        ].to_string(index=False)
    )

    # --------------------------------------------------------
    # Output directory
    # --------------------------------------------------------

    output_dir = (
        MODEL_ROOT
        / category_lower
    )

    # --------------------------------------------------------
    # Configuration
    # --------------------------------------------------------

    config = TrainingConfig(

        dataset_path=str(
            MERGED_BUNDLE
        ),

        label_ids=label_ids,

        labels_csv=str(
            LABELS_CSV
        ),

        model_type="sign_lstm",

        output_dir=str(
            output_dir
        ),

        model_name=f"{category_lower}_lstm",

        batch_size=8,

        epochs=50,

        learning_rate=0.001,

        patience=10,

        random_state=42,
    )

    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    try:

        result = train(config)

        results[category_upper] = result

        print("\n")
        print(
            f"✓ {category_upper} TRAINING COMPLETE"
        )

        print(
            f"Validation accuracy: "
            f"{result['best_val_accuracy']:.4f}"
        )

        print(
            f"Test accuracy: "
            f"{result['test_accuracy']:.4f}"
        )

        # --------------------------------------------------------
        # Per-label accuracy
        # --------------------------------------------------------

        y_true = result["y_true"]
        y_pred = result["y_pred"]

        # Confusion matrix
        cm = confusion_matrix(
            y_true,
            y_pred,
            labels=range(len(label_ids)),
        )

        print("\nPer-label accuracy:")

        for index, label_id in enumerate(label_ids):

            label_row = group[
                group["id"].astype(int) == label_id
            ]

            if label_row.empty:
                label_name = f"ID {label_id}"
            else:
                label_name = label_row.iloc[0]["label"]

            total = cm[index].sum()
            correct = cm[index, index]

            accuracy = (
                correct / total
                if total > 0
                else 0.0
            )

            print(
                f"{label_name:20s}: "
                f"{accuracy:.2%} "
                f"({correct}/{total})"
            )

        print("\nTop confused pairs:")

        confusions = []

        for true_idx in range(len(label_ids)):

            for pred_idx in range(len(label_ids)):

                if true_idx == pred_idx:
                    continue

                count = cm[true_idx, pred_idx]

                if count > 0:
                    confusions.append(
                        (
                            count,
                            label_ids[true_idx],
                            label_ids[pred_idx],
                        )
                    )

                confusions.sort(reverse=True)

                for count, true_id, pred_id in confusions[:5]:

                    true_name = group[
                        group["id"].astype(int) == true_id
                    ].iloc[0]["label"]

                    pred_name = group[
                        group["id"].astype(int) == pred_id
                    ].iloc[0]["label"]

                    print(
                        f"True: {true_name:20s} "
                        f"→ Predicted: {pred_name:20s} "
                        f"| {count}"
                    )

    except Exception as e:

        print("\n")
        print(
            f"✗ {category_upper} TRAINING FAILED"
        )

        print(
            f"Error: {e}"
        )

        results[category_upper] = {
            "error": str(e)
        }



TRAINING CATEGORY: CALENDAR
Labels: [30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41]

Classes:
 id     label
 30   JANUARY
 31  FEBRUARY
 32     MARCH
 33     APRIL
 34       MAY
 35      JUNE
 36      JULY
 37    AUGUST
 38 SEPTEMBER
 39   OCTOBER
 40  NOVEMBER
 41  DECEMBER
Device: cpu
Selected labels: [30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41]

DATASET DIAGNOSTIC
Raw bundle path : C:\Projects\signia-fsl-recognition\data\merged_dataset.pt
Raw bundle size : 2,294 samples
Raw label IDs   : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104] (105 classes)

Selected label IDs : [30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 

In [ ]:
print("\n")
print("=" * 70)
print("ALL CATEGORY TRAINING RESULTS")
print("=" * 70)

for category, result in results.items():

    if "error" in result:

        print(
            f"{category:<20} FAILED"
        )

    else:

        print(
            f"{category:<20} "
            f"Val: {result['best_val_accuracy']:.3f} | "
            f"Test: {result['test_accuracy']:.3f}"
        )